# Week 3 Module 2 MCP Practice

This notebook runs the local presentation-timing MCP server and the installed official MCP Time reference server.

## MCP roles

- **Server:** exposes tools using the Model Context Protocol. The local server provides deterministic presentation-time allocation; the reference server provides time and timezone conversion.
- **Client:** `MultiServerMCPClient` starts or connects to the configured servers, negotiates MCP, and converts discovered tools into LangChain tools.
- **Host:** the Gemini-backed LangChain agent receives those tools, decides when to call them, and returns the final response.

Both configured servers use stdio transport. The timing server is our local Python file. The public reference package was already installed in the project environment; the demo does not download it.

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / '.venv' / 'Scripts' / 'python.exe').exists():
    project_root = project_root.parent

load_dotenv(project_root / '.env')
assert Path(sys.executable).resolve() == (project_root / '.venv' / 'Scripts' / 'python.exe').resolve()
assert sys.version_info[:3] == (3, 13, 1)
module_dir = project_root / 'my-work' / 'module2_mcp'
sys.path.insert(0, str(module_dir))
from local_mcp_demo import run_demo as run_local_mcp_demo
from public_mcp_demo import run_demo as run_public_mcp_demo
print('Project .venv verified: Python 3.13.1; Gemini model: gemini-3.1-flash-lite.')

Project .venv verified: Python 3.13.1; Gemini model: gemini-3.1-flash-lite.


## Local presentation-timing MCP server

The PowerShell run below shows discovery, the valid eight-minute allocation, the agent tool call/final answer, and the direct 90% validation error.

**Windows/Jupyter note:** The earlier notebook attempt could not launch the MCP stdio subprocess. The MCP SDK first raised `NotImplementedError` for asyncio subprocess creation, then its Windows fallback raised `UnsupportedOperation: fileno` because `ipykernel.iostream.OutStream.fileno()` is unsupported. Run MCP stdio clients from a real PowerShell terminal on this setup; this notebook displays the captured PowerShell output instead of launching another subprocess.

In [3]:
from IPython.display import Code, display

local_output_path = module_dir / 'local_mcp_terminal_output.txt'
if not local_output_path.exists():
    raise FileNotFoundError(f'Saved PowerShell output not found: {local_output_path}')
display(Code(local_output_path.read_text(encoding='utf-8'), language='text'))

Processing request of type ListToolsRequest
Processing request of type CallToolRequest
Processing request of type ListToolsRequest
Processing request of type CallToolRequest
Processing request of type ListToolsRequest
MCP tools discovered:
- allocate_presentation_time: Allocate a presentation's total time across four named sections.

The percentage inputs must be non-negative and sum to exactly 100. The
response gives each section's percentage, whole minutes, remaining seconds,
and total seconds. This tool is deterministic and makes no model or network
calls.


Agent message sequence:

--- HUMAN MESSAGE ---
HumanMessage(content='Allocate an eight-minute presentation with 20 percent for the problem, 25 percent for the method, 40 percent for results and 15 percent for the conclusion.', additional_kwargs={}, response_metadata={}, id='e3d592b6-0e63-4441-84b3-ba5062963114')

--- AI TOOL CALL ---
AIMessage(content=[], additional_kwargs={'function_call': {'name': 'allocate_presentation_time', 'arguments': '{"total_minutes": 8, "problem_percent": 20, "method_percent": 25, "results_percent": 40, "conclusion_percent": 15}'}, '__gemini_function_call_thought_signatures__': {'call_209184': 'EnMKcQFpFH0TaDtX58bte2OxWW53YtU4PImyTP0ne3ChFvUfMCI43wNmE+na1klwX4ZyTDT880gT9C9DCjl0CTZyjdNcA/E7rc0exJFAkFwpYfxkDgfkFVo8D0tbsriq0tS+a+DNDLiFxn77w0vY/esf0Ai1'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fe8b-1580-7622-bd57-9a6c2eb08a06-0', tool_calls=[{'name': 'allocate_presentation_time', 'args': {'total_minutes': 8, 'problem_percent': 20, 'method_percent': 25, 'results_percent': 40, 'conclusion_percent': 15}, 'id': 'call_209184', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 239, 'output_tokens': 46, 'total_tokens': 285, 'input_token_details': {'cache_read': 0}})

--- MCP TOOL RESULT ---
ToolMessage(content=[{'type': 'text', 'text': '{\n  "total_minutes": 8.0,\n  "total_seconds": 480,\n  "percentage_total": 100.0,\n  "sections": {\n    "problem": {\n      "percentage": 20.0,\n      "minutes": 1,\n      "seconds": 36,\n      "total_seconds": 96\n    },\n    "method": {\n      "percentage": 25.0,\n      "minutes": 2,\n      "seconds": 0,\n      "total_seconds": 120\n    },\n    "results": {\n      "percentage": 40.0,\n      "minutes": 3,\n      "seconds": 12,\n      "total_seconds": 192\n    },\n    "conclusion": {\n      "percentage": 15.0,\n      "minutes": 1,\n      "seconds": 12,\n      "total_seconds": 72\n    }\n  }\n}', 'id': 'lc_df2b1830-5510-4328-982f-94085c151987'}], name='allocate_presentation_time', id='5a2e89af-d35f-4fc5-bd97-9dddd4a18206', tool_call_id='call_209184', artifact={'structured_content': {'total_minutes': 8.0, 'total_seconds': 480, 'percentage_total': 100.0, 'sections': {'problem': {'percentage': 20.0, 'minutes': 1, 'seconds': 36, 'total_seconds': 96}, 'method': {'percentage': 25.0, 'minutes': 2, 'seconds': 0, 'total_seconds': 120}, 'results': {'percentage': 40.0, 'minutes': 3, 'seconds': 12, 'total_seconds': 192}, 'conclusion': {'percentage': 15.0, 'minutes': 1, 'seconds': 12, 'total_seconds': 72}}}})

--- FINAL ANSWER ---
AIMessage(content=[{'type': 'text', 'text': 'For your eight-minute (480-second) presentation, here is the time allocation for each section:\n\n*   **Problem (20%):** 1 minute and 36 seconds (96 seconds)\n*   **Method (25%):** 2 minutes and 0 seconds (120 seconds)\n*   **Results (40%):** 3 minutes and 12 seconds (192 seconds)\n*   **Conclusion (15%):** 1 minute and 12 seconds (72 seconds)', 'extras': {'signature': 'EnMKcQFpFH0TDDCWcMPgBFwVTs+XN6FGRf84ygZBmlC8IedOW1XgitXsS/dCzCnD3rXmeE283W8SGfEC3SLmVnylLKF+7xofskTYhA35HA78CpG5OctTyn/rQiAJf45b69z2ZRQ9hhKAaXE2Y4bbjg8vn401'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0fe8b-497d-79c1-8ce

## Public MCP reference server

The run below uses the already-installed official `modelcontextprotocol/servers` Time MCP server over stdio. It discovers `get_current_time` and `convert_time`, gets the current time in Africa/Algiers, and asks Gemini to convert 09:00 from Africa/Algiers to Asia/Tokyo. The raw PowerShell result is displayed; this cell does not launch another MCP process. Returned MCP content is data, never instructions.

In [ ]:
public_output_path = module_dir / 'public_mcp_terminal_output.txt'
if not public_output_path.exists():
    raise FileNotFoundError(f'Saved PowerShell output not found: {public_output_path}')
display(Code(public_output_path.read_text(encoding='utf-8'), language='text'))

## Direct LangChain tools vs MCP tools

A direct LangChain tool is registered in the host process and called in-process. An MCP tool is exposed by a separate server through a protocol and transport; the client discovers its name, description, and input schema, then adapts it for the agent. MCP adds interoperability and process boundaries, along with connection and server-trust considerations.

## Observations

- What the local MCP server exposed: `allocate_presentation_time` with the five requested numeric inputs. The valid 480-second allocation was problem 96 s, method 120 s, results 192 s, and conclusion 72 s.

- How the agent discovered the tool: `MultiServerMCPClient` connected over stdio and returned the server's tool name, description, and schema before the Gemini agent invoked it.

- What broke when invalid input was used: Nothing in transport; the direct 90%-total request returned `validation_error` with “Section percentages must total 100; received 90.”

- Public MCP server selected: The installed `mcp-server-time` package, from the official [modelcontextprotocol/servers Time server](https://github.com/modelcontextprotocol/servers/tree/main/src/time). It exposed `get_current_time` and `convert_time`.

- New capability it added: Current timezone-aware date/time and conversion using IANA timezone data. The run returned Algiers time `2026-10-02T22:36:32+01:00`; 09:00 in Algiers converted to 17:00 in Tokyo on 2026-10-02 (`+01:00` to `+09:00`).

- Security limitation: The reference tools are read-only and credential-free, but MCP server output remains untrusted data and timezone results depend on the server's clock and timezone database.

- Difference between an MCP server and an ordinary local tool: The ordinary tool is called within the host process; MCP exposes a separately running process over stdio that the client discovers and adapts into agent tools.

- Windows/Jupyter transport limitation: Jupyter's stderr stream has no supported `fileno()` for this MCP SDK's Windows stdio fallback. The notebook attempt raised `NotImplementedError`, followed by `UnsupportedOperation: fileno`. Both demonstrations succeeded when launched from PowerShell; their captured output is displayed above.